## Temperature and Sampling

After an LLM calculates how likely every possible next token is, it still has to actually PICK one. Temperature and sampling parameters control how that final pick is made — and they are the reason the exact same prompt can produce different wording across runs.

## How it actually works

**Recall from the previous notebook — the model produces a probability distribution:**
For every possible next token, the model calculates a probability. Some tokens might
get a 40% likelihood, others 0.001%, across the entire vocabulary. The model itself
does not "choose" anything — it produces this whole distribution. A separate sampling
step decides what to actually pick from it.

**Temperature — controlling how "sharp" or "flat" that distribution is before picking:**
- **Temperature = 0** — always pick the single highest-probability token. Fully deterministic,
  same input produces the same output every time (directly connects to the
  deterministic vs non-deterministic notebook later in this folder).
- **Low temperature (e.g. 0.2-0.3)** — strongly favours the most likely tokens, but
  occasionally allows a slightly less likely one through. Mostly consistent, slightly varied.
- **Higher temperature (e.g. 0.8-1.0+)** — flattens the distribution, giving lower-probability
  tokens a more meaningful chance of being picked. More varied, more "creative" sounding,
  but also more likely to wander or produce less focused output.

**What temperature is mathematically doing (without needing the full equation):**
Temperature reshapes the probability distribution before sampling — low temperature
exaggerates the gap between likely and unlikely tokens (making the model more confident
in its top choices), high temperature compresses that gap (making lower-probability
tokens more competitive with the top choice).

**Top-p (nucleus sampling) — a different lever, often used alongside temperature:**
Instead of reshaping the whole distribution, top-p restricts the selection pool to only
the smallest set of tokens whose combined probability reaches some threshold (e.g. 0.9),
then samples from just that restricted pool. This avoids ever picking from the very
long tail of extremely unlikely tokens, regardless of temperature setting.

**Top-k — a simpler, related lever:**
Restricts the selection pool to only the K most likely tokens (e.g. the top 40),
then samples from just those, ignoring everything else regardless of how the
probabilities are shaped.

## Where it breaks or gets dangerous

**Choosing the wrong temperature for the task is a common, avoidable mistake:**
High temperature for a task requiring precision — extracting a specific number from
a document, generating valid JSON, answering a factual question — increases the risk
of an unnecessary, avoidable error. There is no creative benefit to gain in those tasks,
only downside risk from added randomness. Low temperature for genuinely creative tasks
— brainstorming content ideas, generating varied examples — can produce repetitive,
flat output that lacks the variety the task actually benefits from.

**Temperature 0 is not the same as "always correct," it is "always the same":**
A common misconception: setting temperature to 0 makes the model more accurate.
It does not change the model's underlying knowledge or reasoning quality — it only
removes randomness in which token gets picked when multiple options are close in
probability. If the model's TOP choice is wrong, temperature 0 will confidently and
consistently produce that wrong answer every single time, with no variation that
might have occasionally landed on something better.

**Testing and evaluation get genuinely harder with higher temperature:**
This connects directly to the "deterministic vs non-deterministic" testing problem —
the higher the temperature, the more output variation you will see across repeated
runs of the identical prompt, which is exactly why production evaluation strategies
need to account for this rather than assuming one run represents typical behaviour.

**Direct, honest check on your own project:**
Your Personal Brand Autopilot's four agents do not currently set an explicit temperature
parameter in the API calls we built — meaning they are running at whatever the
provider's default is. Worth deciding deliberately: should the Quality Scorer (a
fact-and-criteria-driven judgment task) run at a LOWER temperature than the Drafter
(a task that benefits from some creative variation in framing)? Right now, both run
at the same implicit default, which may not be the optimal choice for either task.

In [ ]:
# Temperature and Sampling — illustrating how output varies (or doesn't) with temperature

import anthropic
import os
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

prompt = "Give me one creative tagline for a D365 consulting firm. Just the tagline, nothing else."

def generate_with_temperature(temperature, run_number):
    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=30,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()

print("=== Temperature 0 — run the SAME prompt 3 times ===")
for i in range(1, 4):
    result = generate_with_temperature(0, i)
    print(f"Run {i}: {result}")

print()
print("=== Temperature 1.0 — run the SAME prompt 3 times ===")
for i in range(1, 4):
    result = generate_with_temperature(1.0, i)
    print(f"Run {i}: {result}")

print()
print("At temperature 0, the three runs are typically identical or near-identical —")
print("the model consistently picks its single highest-probability token each time.")
print()
print("At temperature 1.0, the three runs typically differ noticeably —")
print("the model is sampling from a flattened distribution, allowing more variety")
print("in which token gets picked at each step.")

## My D365 analogy

Temperature is similar to how much discretion you give a junior consultant when
filling in a free-text field versus a strictly validated field in a configuration.

At "temperature 0" — a strictly validated dropdown with one obviously correct value
given the context — the same input always produces the same selection, every time,
by every person, with zero variation. Reliable, but only as good as whether that
single obvious choice was actually the right one for this specific situation.

At "high temperature" — an open free-text comment field where the consultant has
real discretion in how they phrase something — you get more variety and sometimes
more useful nuance captured, but also more risk of an unusual, less appropriate
phrasing slipping through occasionally, simply because more options were genuinely
on the table.

The skill is not picking one setting forever. It is recognising which specific field,
which specific task, benefits from tight consistency versus which one benefits from
some controlled discretion — and setting the constraint level deliberately for each,
rather than applying one blanket policy everywhere out of habit.

## LinkedIn post idea

People sometimes assume turning an AI model's "creativity setting" down to zero
makes it more accurate. It does not. It makes it more CONSISTENT.

Temperature controls how much randomness goes into picking the next word, not how
much the model actually knows or how well it reasons. At temperature zero, the model
picks its single most likely next word every time, with no variation. If that top
choice happens to be wrong, you will get that same wrong answer, confidently, every
single time you ask — with no chance of occasionally landing on something better.

This matters more than it sounds for anyone building real AI systems. The right setting
depends entirely on the task. A system extracting a specific number from a document
benefits from low temperature — no upside in randomness there, only downside risk.
A system brainstorming content ideas benefits from some real variation, or every
output starts sounding the same.

I spent years deciding which D365 fields deserved tight, validated control and
which ones deserved free-text discretion. Turns out that exact same judgment call
shows up again, almost unchanged, in how you configure an AI system.